# S6 · First real dosimetry and go/no-go 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/sct-range/blob/main/notebooks/s6_dosimetry.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hashirama21/sct-range/blob/main/notebooks/s6_dosimetry.ipynb)

Week 6 turns the range proxy of weeks 1–5 into dose, on a frozen subset of 20 development cases:

| Step | What | Plan item |
|---|---|---|
| 1 | IMPT (3 beams) and IMRT (7 beams) plans optimised on the reference CT with pyRadPlan | proton (and photon) plans on 20 cases |
| 2 | the same weights recomputed on the bulk-density baseline and on R1–R6 (every seed) | recomputation on R1–R6 |
| 3 | gamma pass rates, target DVH points, distal R80 shift of every proton beam along every ray | |
| 4 | WET error to the distal target edge against the gamma failure rate (cases) and against the R80 shift (rays) | validation of the proxy |
| 5 | the S4 range verdict combined with the proton gamma test, rule fixed in `configs/s6_preregistration.json` | **go/no-go 2** |

SynthRAD2025 releases no PTV, so the target is the 25 mm sphere at the body centroid used since week 1, and voxels
outside the body mask are set to air in the reference CT and in every sCT: only errors inside the patient reach the
dose. pyRadPlan is the dose engine of the SynthRAD2025 evaluation; its default HU → SPR table is the one used by the
range projector, so the proxy and the dose see the same stopping powers.

The verdict needs the S4–S5 runs (`outputs/s4/runs/seed<k>/<arm>`) and their report (`outputs/s4/s4_report.json`).
Dose calculation runs on the CPU (no GPU needed); each case takes several minutes per source, so the 20 cases are
resumable: every finished case is stored and reused.

The notebook runs on the **real SynthRAD2025 training data** (Zenodo record 15373853), read from its
Hugging Face mirror [developper669/synthrad2025](https://huggingface.co/datasets/developper669/synthrad2025)
(tag `v2`). `SYNTHRAD2025_ROOT` points to the local copy (default `data/synthrad2025` in the repository,
ignored by git); missing tasks are fetched first (`sctrange.download`, parallel and resumable, only the
requested task; `SCT_DOWNLOAD=0` forbids it and stops with an error instead).

### Setup · install `sctrange` from the repository

Run this cell first. Inside a clone of [hashirama21/sct-range](https://github.com/hashirama21/sct-range) it installs that
clone in editable mode (`pip install -e ".[dose]"`), so the notebook always runs the repository's code, local edits
included. Opened on its own, from the Colab or Kaggle badge or on another machine, it first downloads the repository
from GitHub (no git needed) to `/content/sct-range` on Colab, `/kaggle/tmp/sct-range` on Kaggle (so that
`/kaggle/working` only receives outputs) or `./sct-range` elsewhere, then installs it. On Kaggle, enable
*Settings → Internet* first. `SCTRANGE_REF` selects a branch, tag or commit (default `main`), `SCTRANGE_HOME` the
destination; pip only runs when `sctrange` is missing or installed from elsewhere (`SCTRANGE_REINSTALL=1` forces it).

In [ ]:
import importlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

# OpenBLAS 0.3.27 can corrupt multithreaded matrix products, and pyRadPlan (S6) optimises plans and computes dose with
# NumPy: one BLAS thread per process. The variable only acts before NumPy is loaded; threadpoolctl covers a later cell.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
if "numpy" in sys.modules:
    try:
        from threadpoolctl import threadpool_limits

        BLAS_LIMIT = threadpool_limits(1, user_api="blas")
    except ImportError:
        print("NumPy was loaded before this cell: restart the kernel so that OPENBLAS_NUM_THREADS=1 applies")

REPO_URL = "https://github.com/hashirama21/sct-range"
REPO_REF = os.environ.get("SCTRANGE_REF", "main")
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()


def find_repo(start: Path) -> Path | None:
    return next((p for p in (start, *start.parents) if (p / "pyproject.toml").exists() and (p / "src" / "sctrange").is_dir()),
                None)


def download_repo(destination: Path) -> Path:
    """Fetch ``REPO_REF`` of the GitHub repository as a zip archive and unpack it to ``destination``."""
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f"{REPO_URL}/archive/{REPO_REF}.zip", timeout=120) as response:
                archive = zipfile.ZipFile(io.BytesIO(response.read()))
            break
        except OSError as error:
            if attempt == 3:
                raise
            print(f"download failed ({error}); retrying")
            time.sleep(5 * (attempt + 1))
    top = archive.namelist()[0].split("/")[0]
    staging = destination.with_name(destination.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():  # an earlier, interrupted attempt or another folder
        if any(destination.iterdir()):
            raise RuntimeError(f"{destination} exists and is not a clone of {REPO_URL}; remove it or set SCTRANGE_HOME")
        destination.rmdir()
    (staging / top).rename(destination)
    shutil.rmtree(staging, ignore_errors=True)
    return destination


REPO_ROOT = find_repo(Path.cwd())
if REPO_ROOT is None:
    default_home = (Path("/content/sct-range") if ON_COLAB else Path("/kaggle/tmp/sct-range") if ON_KAGGLE
                    else Path.cwd() / "sct-range")
    REPO_ROOT = Path(os.environ.get("SCTRANGE_HOME", default_home)).resolve()
    if find_repo(REPO_ROOT) is None:
        print(f"downloading {REPO_URL} @ {REPO_REF} to {REPO_ROOT}")
        download_repo(REPO_ROOT)
    os.chdir(REPO_ROOT / "notebooks")
SOURCE_DIR = (REPO_ROOT / "src").resolve()


def installed_from_repo() -> bool:
    spec = importlib.util.find_spec("sctrange")
    return spec is not None and spec.origin is not None and Path(spec.origin).resolve().is_relative_to(SOURCE_DIR)


if os.environ.get("SCTRANGE_REINSTALL") == "1" or not installed_from_repo():
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{REPO_ROOT}[dose]"]
    for attempt in range(1, 4):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f"pip install failed (attempt {attempt}/3):\n{result.stderr[-3000:]}")
        time.sleep(10 * attempt)
    else:
        raise RuntimeError("pip could not install sctrange; see the messages above (network? Kaggle: enable Internet)")
# an editable install is only picked up by new processes; make it visible to this kernel too
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module in [name for name in sys.modules if name == "sctrange" or name.startswith("sctrange.")]:
    del sys.modules[module]
importlib.invalidate_caches()

import sctrange

assert Path(sctrange.__file__).resolve().is_relative_to(SOURCE_DIR), f"sctrange imported from {sctrange.__file__}"
print(f"sctrange {sctrange.__version__} installed from {REPO_ROOT.name}/src (editable)")

### Parameters

Set them here and run all cells (*Runtime → Run all* on Colab, where this cell is a form). Environment variables of the
same meaning, given in brackets, override them for unattended runs (`jupyter nbconvert --execute`). The verdict assumes
the pre-registered defaults (20 cases, seeds 0, 1, 2); fewer give a quick pass only.

| Platform | Dataset, cache | Outputs |
|---|---|---|
| local | `data/synthrad2025`, `outputs/` | `outputs/` |
| Colab | `/content/data`, `/content/cache` | Google Drive `MyDrive/sct-range/outputs` (`USE_DRIVE`) |
| Kaggle | `/kaggle/tmp/data`, `/kaggle/tmp/cache` | `/kaggle/working/outputs` |

`WORKERS` doses that many cases in parallel processes (pyRadPlan is single-threaded for most of the work); each worker
needs about 4 GB of memory on a real case.

In [ ]:
# @title Parameters { display-mode: "form" }
TASK = "1"  # @param ["1", "2"]  [SCT_TASK]
DOWNLOAD = True  # @param {type:"boolean"}  fetch missing data from Hugging Face [SCT_DOWNLOAD]
N_CASES = 20  # @param {type:"integer"}  dosimetry cases [SCT_DOSE_CASES]
SEEDS = "0,1,2"  # @param {type:"string"}  S4 seeds to recompute [SCT_SEEDS]
WORKERS = 1  # @param {type:"integer"}  cases dosed in parallel [SCT_WORKERS]
USE_DRIVE = True  # @param {type:"boolean"}  Colab only

In [ ]:
import json
import os
import warnings
from dataclasses import asdict

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sctrange.diagnostics import paired_test
from sctrange.dosimetry import (DoseConfig, average_seeds, decide_go_no_go_2, engine_checks, load_selection,
                                proxy_validation, run_dosimetry, select_cases, split_source, write_selection)
from sctrange.download import ensure_dataset, usable_records
from sctrange.data import discover_cases
from sctrange.plotting import apply_style, plot_dose_slices, plot_gamma_by_source, plot_proxy_validation
from sctrange.runtime import frozen_split_path, resolve_paths
from sctrange.split import load_split, split_hash

In [ ]:
PATHS = resolve_paths(REPO_ROOT, USE_DRIVE)
TASK = os.environ.get("SCT_TASK", TASK)
DOWNLOAD = os.environ.get("SCT_DOWNLOAD", "1" if DOWNLOAD else "0") != "0"
N_CASES = int(os.environ.get("SCT_DOSE_CASES", N_CASES))
SEEDS = tuple(int(s) for s in os.environ.get("SCT_SEEDS", SEEDS).split(","))
WORKERS = int(os.environ.get("SCT_WORKERS", WORKERS))

OUTPUT_DIR = PATHS.output_root / "s6"
FIGURE_DIR = OUTPUT_DIR / "figures"
S4_RUNS = Path(os.environ.get("SCT_S4_RUNS_ROOT", PATHS.output_root / "s4" / "runs"))
FROZEN_SPLIT_PATH = frozen_split_path(REPO_ROOT, PATHS)
PREREGISTRATION_PATH = REPO_ROOT / "configs" / "s6_preregistration.json"

prereg_bytes = PREREGISTRATION_PATH.read_bytes()
PREREG = json.loads(prereg_bytes)
PREREG_SHA256 = __import__("hashlib").sha256(prereg_bytes).hexdigest()
DOSE = DoseConfig.from_preregistration(PREREG)
ARMS = tuple(PREREG["arms"])
ARM_ORDER = ("bulk density",) + ARMS

DATA_ROOT = ensure_dataset(PATHS.data_root, (TASK,), download=DOWNLOAD)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
pd.set_option("display.width", 160)
display(pd.Series(PATHS.describe(REPO_ROOT), name="location").to_frame())
print(f"task {TASK} · cases {N_CASES} · seeds {SEEDS} · workers {WORKERS} · S4 runs {S4_RUNS}")
if (N_CASES, SEEDS) != (PREREG["cases"]["n_cases"], tuple(PREREG["seeds"])):
    print("non-default cases or seeds: quick pass only, the verdict is not the pre-registered one")

## 1 · Frozen split, pre-registration and dosimetry subset

The split of week 1 is verified; test cases are never loaded. The 20 cases are drawn once from the development pool
(region-stratified, hashed order within region, cases with an out-of-fold R3 prediction), written with their SHA-256
and reused on every run: a rerun that would select different cases stops instead of silently reselecting.

In [ ]:
if not FROZEN_SPLIT_PATH.exists():
    raise FileNotFoundError(f"{FROZEN_SPLIT_PATH} is missing: run the S1 notebook once on the real data to freeze it")
split = load_split(FROZEN_SPLIT_PATH)
development_ids = set(split["development"])
assert not development_ids & set(split["test"])
development, excluded = usable_records(DATA_ROOT, [r for r in discover_cases(DATA_ROOT)
                                                  if r.case_id in development_ids and r.task == TASK])

reference_prediction = lambda r: S4_RUNS / "seed0" / PREREG["reference_arm"] / "predictions" / f"{r.case_id}.mha"
if not any(reference_prediction(r).exists() for r in development):
    raise RuntimeError(f"no out-of-fold {PREREG['reference_arm']} prediction under {S4_RUNS}: run S4–S5 first "
                       "(notebooks/s4_regression_arms.ipynb)")
selection_path = OUTPUT_DIR / "dosimetry_cases.json"
selection_meta = {"task": TASK, "seed": PREREG["cases"]["seed"], "n_cases": N_CASES, "split_sha256": split_hash(split)}
chosen_ids = select_cases(development, N_CASES, PREREG["cases"]["seed"], lambda r: reference_prediction(r).exists())
selection_sha256 = write_selection(chosen_ids, selection_path, selection_meta)
chosen = [r for r in development if r.case_id in set(load_selection(selection_path)["case_ids"])]

display(pd.Series({"split sha256": split_hash(split), "pre-registration sha256": PREREG_SHA256,
                   "dosimetry selection sha256": selection_sha256, "cases": len(chosen)}, name="value").to_frame())
display(pd.Series([r.region for r in chosen]).value_counts().rename("cases").rename_axis("region").to_frame().T)

## 2 · Dose-engine checks

Two checks on a water cylinder, before any patient:

- **beam direction**: a single-beam proton plan at each plan angle must deposit dose upstream of the target along the
  projector direction and almost none downstream. This validates the angle conversion
  `gantry = (360 − angle) mod 360` between the projector (angle 90 travels along +x) and pyRadPlan (gantry 90 travels
  along −x), without which the proxy and the dose would look along different rays;
- **identity**: recomputing the optimised weights on the planning CT reproduces the reference dose (γ 100 %).

In [ ]:
checks = engine_checks(DOSE)
display(checks.round(3))
assert checks["direction_ok"].all() and (checks["identity_gamma_2_2"] >= 99.99).all(), "dose-engine check failed"

## 3 · Plans and recomputation

For each case: optimise both plans on the reference CT, recompute them on the reference CT (the reference dose, same
engine path as every sCT) and on every source. Per source and modality: γ 2 %/2 mm and 1 %/1 mm (global, 10 % cutoff),
target mean, D95 and D2 differences (% of the 60 Gy prescription), mean absolute dose difference above the cutoff, and
the range proxy on the proton beams (WET error from entry to the distal target edge). Metrics are then averaged over
seeds per case and arm, as in S4. Every case is written as soon as it is dosed, so an interrupted session resumes with
the cases that remain.

In [ ]:
per_seed, rays = run_dosimetry(chosen, S4_RUNS, OUTPUT_DIR, DOSE, ARMS, SEEDS, workers=WORKERS)
per_seed.to_csv(OUTPUT_DIR / "s6_dose_per_seed.csv", index=False)
rays.to_csv(OUTPUT_DIR / "s6_rays.csv", index=False)
cases = average_seeds(per_seed)
cases.to_csv(OUTPUT_DIR / "s6_dose.csv", index=False)

order = [s for s in ARM_ORDER if s in set(cases["source"])]
columns = ["mae_hu", "proxy_rms_mm", "proxy_cvar_mm", "proton_gamma_2_2", "proton_gamma_1_1", "photon_gamma_2_2",
           "photon_gamma_1_1", "proton_target_d95_diff_pct", "proton_dose_mae_pct"]
print(f"{cases['case_id'].nunique()} cases · sources {order} · {len(rays)} proton rays")
display(cases.groupby("source")[columns].median().reindex(order).round(3))

Central target slice of the first case: reference proton dose, then the dose difference of a few sources. Range errors
show as distal shifts behind the target; the photon plan of the same case barely changes.

In [ ]:
example = chosen[0].case_id
slices = dict(np.load(OUTPUT_DIR / "slices" / f"{example}.npz"))
shown = [s for s in ("bulk density", "R1@0", "R3@0", "R5@0") if f"proton/{s}" in slices]
for modality in ("proton", "photon"):
    plot_dose_slices(slices, shown, modality, DOSE.prescription_gy).savefig(
        FIGURE_DIR / f"dose_{modality}_{example}.png", bbox_inches="tight")
    plt.show()
plot_gamma_by_source(cases, order).savefig(FIGURE_DIR / "gamma_by_source.png", bbox_inches="tight")
plt.show()

## 4 · Does the range proxy track dose?

Pre-registered criteria (`proxy` block): across cases and sources, the RMS WET error to the distal target edge on the
plan beams correlates with the proton γ 2 %/2 mm failure rate (Spearman ≥ 0.5, case-bootstrap 95 % interval above 0)
**and still does when MAE is held fixed** (partial Spearman, interval above 0); per ray, the WET error predicts the
distal R80 shift (Spearman between ΔWET and −ΔR80 ≥ 0.7). The Theil–Sen slope of ΔR80 on ΔWET is expected near
−1/SPR at the distal edge (about −1 in soft tissue). The MAE-only Spearman is shown for contrast, and the case Spearman without the bulk-density baseline (`arms_only_case_spearman`, secondary: the
baseline is far from every arm and can carry the pooled correlation on its own).

In [ ]:
validation = proxy_validation(cases, rays, PREREG, n_boot=2000, seed=0)
display(pd.Series(validation, name="value").to_frame())
plot_proxy_validation(cases, rays).savefig(FIGURE_DIR / "proxy_validation.png", bbox_inches="tight")
plt.show()
if not rays.empty:
    by_source = split_source(rays).groupby("arm").apply(
        lambda f: pd.Series({"rays": len(f), "median ΔWET (mm)": f["delta_wet_mm"].median(),
                             "median ΔR80 (mm)": f["delta_r80_mm"].median(),
                             "Spearman(ΔWET, −ΔR80)": f["delta_wet_mm"].corr(-f["delta_r80_mm"], method="spearman")}),
        include_groups=False)
    display(by_source.reindex([s for s in ARM_ORDER if s in by_source.index]).round(3))

## 5 · Go/no-go 2

The rule of `configs/s6_preregistration.json`, applied unchanged:

| Verdict | Condition |
|---|---|
| GO | a method arm beats R3 **and** R4 on target CVaR90 in S4 (MAE condition met) **and** beats R3 on proton γ 2 %/2 mm here |
| RANGE GAIN WITHOUT DOSE CONFIRMATION | it beats R3 and R4 in S4, but no method arm beats R3 on proton γ |
| RIGHT ESTIMATOR SUFFICES | some method arm beats R3 in S4, none beats R4: the gain is a tissue-level bias |
| NO-GO | no method arm beats R3 in S4 |

γ tests are one-sided paired Wilcoxon over the dosimetry cases, Holm over the four comparisons. Secondary results are
reported whatever the verdict.

In [ ]:
s4_report_path = PATHS.output_root / PREREG["s4_report"]
s4_report = json.loads(s4_report_path.read_text()) if s4_report_path.exists() else None
if s4_report is None:
    print(f"{s4_report_path} not found: run notebooks/s4_regression_arms.ipynb first")
decision = decide_go_no_go_2(cases, s4_report, PREREG)
display(pd.Series({k: v for k, v in decision.items() if k not in ("arms", "raw_p_values")}, name="value").to_frame())
display(pd.DataFrame(decision["arms"]).T)

secondary = {
    "proton γ 2/2, R1 − R3": paired_test(cases, "proton_gamma_2_2", "R1", "R3"),
    "proton γ 2/2, R4 − R3": paired_test(cases, "proton_gamma_2_2", "R4", "R3"),
    "photon γ 2/2, R5 − R3": paired_test(cases, "photon_gamma_2_2", "R5", "R3"),
}
display(pd.DataFrame(secondary).T.round(4))
dvh = ["proton_target_mean_diff_pct", "proton_target_d95_diff_pct", "proton_target_d2_diff_pct",
       "photon_target_d95_diff_pct"]
display(cases.pivot_table(index="region", columns="source", values="proton_target_d95_diff_pct", aggfunc="median")
        .reindex(columns=order).round(2).rename_axis(columns="median proton D95 difference (% of prescription)"))
display(cases.groupby("source")[dvh].median().reindex(order).round(2))

## 6 · Report

In [ ]:
report = {
    "task": TASK,
    "seeds": list(SEEDS),
    "split_sha256": split_hash(split),
    "preregistration_sha256": PREREG_SHA256,
    "selection_sha256": selection_sha256,
    "cases": [r.case_id for r in chosen],
    "dose_config": asdict(DOSE),
    "engine_checks": checks.to_dict(orient="records"),
    "proxy_validation": validation,
    "decision": decision,
    "secondary": secondary,
    "medians_by_source": cases.groupby("source")[columns].median().to_dict(orient="index"),
}
(OUTPUT_DIR / "s6_report.json").write_text(json.dumps(report, indent=2, default=float))
status = pd.DataFrame([
    ("Frozen split verified", split_hash(split)[:12], True),
    ("Dosimetry subset frozen", selection_sha256[:12], True),
    ("Dose-engine checks", f"{len(checks)} beams", bool(checks["direction_ok"].all())),
    ("All arms and seeds dosed", f"{per_seed['source'].nunique()} sources",
     all(f"{a}@{s}" in set(per_seed["source"]) for a in ARMS for s in SEEDS)),
    ("Proxy validation", validation["verdict"], validation["verdict"] == "VALIDATED"),
    ("Go/no-go 2", decision["verdict"], decision["verdict"] != "NOT EVALUATED"),
], columns=["item", "detail", "passed"])
display(status)

## What each verdict implies

| Verdict | Next |
|---|---|
| GO | The paper keeps the range-aware loss as its method contribution; S7–S8 compare the generative arm with R6 on tails. |
| RANGE GAIN WITHOUT DOSE CONFIRMATION | Report the range gain with the γ effect size and its interval; 20 cases may be underpowered for γ, so the S11 test set (opened once) decides. |
| RIGHT ESTIMATOR SUFFICES | The paper becomes "the right estimator suffices": theory (Results A–C), R1 vs R3 vs R4, tails via R6 and the generative arm. |
| NO-GO | The range loss is dropped as a contribution; the paper rests on estimator choice, tails and projected CRPS. |

Whatever the verdict, a **VALIDATED** proxy lets the later weeks (S9–S10 ablations, test set) report range metrics
without recomputing dose for every variant; **PARTIAL** or **NOT VALIDATED** means dose must be recomputed for the
final comparison.